# yolo_lpr_cpp — 検出器の学習 (M7) on Colab

**目的は 1 つ: プレート単体（近接）でも検出できる検出器を作る。**

借り物の検出器 2 つはどちらも車体込みの小さいプレート専用で、実測は

| プレートが画面幅に占める割合 | 5% | 8% | 12% | 20% | 30%以上 |
|---|---|---|---|---|---|
| スコア | 0.757 | 0.826 | 0.695 | 0.069 | **検出なし** |

合成データ（`jlpr gen-det`）はプレート幅を画像幅の 3%〜95% で対数一様に振るので、この穴を埋められる。
合格条件は `tools/eval_det.py` のバケット別 recall（全バケットで高いこと）と、
実写での mAP@0.5 ≥ 0.95・色別 recall ≥ 0.90。

**注意**: 出荷モデルは Ultralytics の yolov8n から転移するので AGPL-3.0 を継承する（承知の上・README 参照）。

In [ ]:
!nvidia-smi -L
!git clone -q https://github.com/yomei-o/yolo_lpr_cpp.git
%cd yolo_lpr_cpp
!pip -q install ultralytics onnx onnxruntime

In [ ]:
# 背景プール: alpr_jp のネガ 4400 枚（実写のパッチ）。車の写真があればそれも足すとよい。
!git clone -q --depth 1 https://github.com/dyama/alpr_jp.git ../alpr_jp
!python tools/fetch_fonts.py
!g++ -std=c++20 -O2 -fopenmp -Ipure -Ipure/third_party pure/jlpr.cpp -o jlpr

In [ ]:
# 学習用と検証用を別 seed で生成（検証は学習に出ていない乱数列から）
!./jlpr gen-det --out data/det_train --count 12000 --seed 4242 --imgsz 640 --bg ../alpr_jp/train/neg --quiet
!./jlpr gen-det --out data/det_val   --count 1200  --seed 991  --imgsz 640 --bg ../alpr_jp/train/neg --quiet
!ls data/det_train/images | wc -l && ls data/det_val/images | wc -l

In [ ]:
# 学習前のベースライン（借り物の検出器を同じ検証セットで測る）。これを超えるのが目標。
!python tools/eval_det.py --data data/det_val --det models/plate_det_pyj320.onnx --det-kind v8 --limit 300

In [ ]:
!python tools/train_det.py --data data/det_train --val data/det_val \
  --epochs 40 --imgsz 640 --batch 32 \
  --export models/plate_det_v8n.onnx --export-imgsz 320 --export-imgsz 640

In [ ]:
# 出力は NMS 抜き（C++/WASM の infer_v8.hpp が decode+NMS を持つ）。素の export は box が cxcywh。
!python tools/eval_det.py --data data/det_val --det models/plate_det_v8n_640.onnx --det-kind v8 --fmt cxcywh --limit 300
!python tools/eval_det.py --data data/det_val --det models/plate_det_v8n_320.onnx --det-kind v8 --fmt cxcywh --limit 300

In [ ]:
# 実写 1 枚でパイプライン全体を通し、近接でも取れるか（context_test）を確認
!python tools/infer.py --img assets/tokyu-bus-yokohama200ka3591.jpg \
  --det models/plate_det_v8n_640.onnx --det-kind v8 --conf 0.3
!python tools/context_test.py --det models/plate_det_v8n_640.onnx --det-kind v8

In [ ]:
from google.colab import files
files.download('models/plate_det_v8n_320.onnx')
files.download('models/plate_det_v8n_640.onnx')